<a href="https://colab.research.google.com/github/Astreele/ATLT/blob/main/ComfyUI_Search_And_Run_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 ComfyUI Model Finder — Temporary Colab

Run **ComfyUI** in Google Colab, search/download models from **CivitAI** and **Hugging Face**, then open ComfyUI from a public link.

This notebook is built for public GitHub use:

- No Google Drive required.
- Files are temporary and disappear when Colab resets.
- No API keys are saved in the notebook.
- Use the search picker first. Manual model lines are kept as an advanced option only.

## Before starting

Go to:

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

Then run the steps from top to bottom.

## Browser note

The public ComfyUI link usually works best in **Chrome or Chromium-based browsers**. If the page opens but does not load correctly in Firefox, try Chrome.

# 🔎 Step 1 — Search and pick models

Use this first.

1. Choose **CivitAI** or **Hugging Face**.
2. Type the model name, model URL, repo URL, model ID, or repo ID.
3. Click **Search**.
4. Pick the exact result from the dropdown.
5. Pick the version/file/folder.
6. Click **Add selected model to MODELS**.

You can add multiple models. After adding everything, run **Step 2 — Check model links**.

In [ ]:
#@title 🔎 Step 1 — Search and pick models { display-mode: "form" }

import re
import json
import urllib.parse
import requests
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

if "MODELS" not in globals():
    MODELS = ""

def _clean_text(x):
    return str(x or "").strip()

def _shorten(x, n=95):
    x = _clean_text(x)
    return x if len(x) <= n else x[:n-3] + "..."

def _safe_label(x):
    return _clean_text(x).replace("|", "-")

def _normalize_search_text(x):
    x = _clean_text(x).lower()
    return re.sub(r"[^a-z0-9]+", "", x)

def _slugify(x):
    x = _clean_text(x).lower()
    return re.sub(r"[^a-z0-9]+", "-", x).strip("-")

# -------------------------
# CivitAI helpers
# -------------------------

def _extract_civitai_model_id(value):
    value = _clean_text(value)
    m = re.search(r"civitai\.com/models/(\d+)", value)
    if m:
        return m.group(1)
    if value.isdigit():
        return value
    return None

def _civitai_api_get_model(model_id):
    headers = {
        "User-Agent": "Mozilla/5.0 Colab ComfyUI Model Search",
        "Accept": "application/json",
    }
    r = requests.get(f"https://civitai.com/api/v1/models/{model_id}", headers=headers, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"CivitAI model lookup failed: HTTP {r.status_code}\n{r.text[:500]}")
    return r.json()

def _civitai_search_once(query, limit=100):
    params = {"query": query, "limit": limit, "nsfw": "true"}
    headers = {
        "User-Agent": "Mozilla/5.0 Colab ComfyUI Model Search",
        "Accept": "application/json",
    }
    r = requests.get("https://civitai.com/api/v1/models", params=params, headers=headers, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"CivitAI search failed: HTTP {r.status_code}\n{r.text[:500]}")
    data = r.json()
    return data.get("items") or []

def _stats_text(model):
    stats = model.get("stats") or {}
    downloads = stats.get("downloadCount") or stats.get("downloadCountAllTime") or model.get("downloads") or 0
    likes = stats.get("thumbsUpCount") or model.get("likes") or 0
    try:
        downloads = int(downloads)
    except Exception:
        downloads = 0
    try:
        likes = int(likes)
    except Exception:
        likes = 0
    return f"downloads {downloads:,}, likes {likes:,}"

def _civitai_search(query, limit=100):
    query = _clean_text(query)
    model_id = _extract_civitai_model_id(query)

    if model_id:
        return [_civitai_api_get_model(model_id)]

    query_variants = [query]
    spaced = re.sub(r"[-_]+", " ", query).strip()
    dashed = _slugify(query)
    compact_words = " ".join(query.split())

    for q in [spaced, dashed, compact_words]:
        if q and q not in query_variants:
            query_variants.append(q)

    found = {}
    for q in query_variants:
        try:
            for item in _civitai_search_once(q, limit=limit):
                mid = item.get("id")
                if mid is not None:
                    found[mid] = item
        except Exception:
            pass

    items = list(found.values())
    q_norm = _normalize_search_text(query)
    q_slug = _slugify(query)

    def score(model):
        name = model.get("name") or ""
        name_norm = _normalize_search_text(name)
        name_slug = _slugify(name)
        model_id = str(model.get("id") or "")

        s = 0
        if model_id == query:
            s += 100000
        if name_norm == q_norm:
            s += 90000
        if name_slug == q_slug:
            s += 85000
        if q_norm and name_norm.startswith(q_norm):
            s += 70000
        if q_slug and name_slug.startswith(q_slug):
            s += 70000
        if q_norm and q_norm in name_norm:
            s += 50000
        if q_slug and q_slug in name_slug:
            s += 50000

        query_tokens = [t for t in re.split(r"[^a-z0-9]+", query.lower()) if t]
        name_low = name.lower()
        if query_tokens:
            covered = sum(1 for t in query_tokens if t in name_low)
            s += covered * 5000
            if covered == len(query_tokens):
                s += 12000

        stats = model.get("stats") or {}
        downloads = int(stats.get("downloadCount") or stats.get("downloadCountAllTime") or 0)
        likes = int(stats.get("thumbsUpCount") or 0)
        s += min(downloads, 2_000_000) / 2000
        s += min(likes, 100_000) / 20
        return s

    return sorted(items, key=score, reverse=True)

def _civitai_versions(model):
    return model.get("modelVersions") or []

def _version_label(model, version):
    version_name = version.get("name") or f"version {version.get('id')}"
    version_id = version.get("id")
    files = version.get("files") or []
    primary = [f for f in files if f.get("primary")] or files
    size_kb = None
    file_name = ""
    if primary:
        chosen = sorted(primary, key=lambda f: f.get("sizeKB") or 0, reverse=True)[0]
        size_kb = chosen.get("sizeKB")
        file_name = chosen.get("name") or ""
    size_txt = ""
    if size_kb:
        size_txt = f" | {size_kb/1024/1024:.2f} GB"
    return f"{version_name} | version id {version_id}{size_txt} | {_shorten(file_name, 45)}"

def _civitai_copy_line(model, version, folder):
    model_name = _safe_label(model.get("name") or "CivitAI model")
    version_name = _safe_label(version.get("name") or f"version {version.get('id')}")
    version_id = version.get("id")
    label = f"{model_name} — {version_name}"
    return f"{folder} | https://civitai.com/api/download/models/{version_id} | {label}"

# -------------------------
# Hugging Face helpers
# -------------------------

def _extract_hf_repo_id(value):
    value = _clean_text(value)

    if value.startswith("https://huggingface.co/"):
        parsed = urllib.parse.urlparse(value)
        parts = [p for p in parsed.path.split("/") if p]
        # /owner/repo or /owner/repo/tree/main or /owner/repo/blob/main/file
        if len(parts) >= 2:
            return "/".join(parts[:2])

    # owner/repo
    if re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$", value):
        return value

    return None

def _hf_api_model_info(repo_id):
    headers = {
        "User-Agent": "Mozilla/5.0 Colab ComfyUI Model Search",
        "Accept": "application/json",
    }
    r = requests.get(f"https://huggingface.co/api/models/{repo_id}", headers=headers, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Hugging Face model lookup failed: HTTP {r.status_code}\n{r.text[:500]}")
    return r.json()

def _hf_search_once(query, limit=100):
    params = {
        "search": query,
        "limit": limit,
        "full": "true",
        "sort": "downloads",
        "direction": "-1",
    }
    headers = {
        "User-Agent": "Mozilla/5.0 Colab ComfyUI Model Search",
        "Accept": "application/json",
    }
    r = requests.get("https://huggingface.co/api/models", params=params, headers=headers, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Hugging Face search failed: HTTP {r.status_code}\n{r.text[:500]}")
    return r.json() or []

def _hf_search(query, limit=100):
    query = _clean_text(query)
    repo_id = _extract_hf_repo_id(query)

    # Direct HF URL or owner/repo should win immediately.
    if repo_id:
        return [_hf_api_model_info(repo_id)]

    query_variants = [query]
    spaced = re.sub(r"[-_]+", " ", query).strip()
    dashed = _slugify(query)
    compact_words = " ".join(query.split())

    for q in [spaced, dashed, compact_words]:
        if q and q not in query_variants:
            query_variants.append(q)

    found = {}
    for q in query_variants:
        try:
            for item in _hf_search_once(q, limit=limit):
                rid = item.get("modelId") or item.get("id")
                if rid:
                    found[rid] = item
        except Exception:
            pass

    items = list(found.values())
    q_norm = _normalize_search_text(query)
    q_slug = _slugify(query)

    def score(model):
        repo_id = model.get("modelId") or model.get("id") or ""
        repo_low = repo_id.lower()
        repo_norm = _normalize_search_text(repo_id)
        repo_slug = _slugify(repo_id)
        repo_name = repo_id.split("/")[-1]
        repo_name_norm = _normalize_search_text(repo_name)
        repo_name_slug = _slugify(repo_name)

        s = 0

        # Exact repo/name/slug matches beat popularity.
        if repo_low == query.lower():
            s += 100000
        if repo_norm == q_norm:
            s += 95000
        if repo_name_norm == q_norm:
            s += 90000
        if repo_slug == q_slug:
            s += 85000
        if repo_name_slug == q_slug:
            s += 85000

        # Prefix/contains
        if q_norm and repo_name_norm.startswith(q_norm):
            s += 75000
        if q_norm and repo_norm.endswith(q_norm):
            s += 70000
        if q_norm and q_norm in repo_name_norm:
            s += 55000
        if q_norm and q_norm in repo_norm:
            s += 45000

        query_tokens = [t for t in re.split(r"[^a-z0-9]+", query.lower()) if t]
        repo_text = repo_id.lower()
        if query_tokens:
            covered = sum(1 for t in query_tokens if t in repo_text)
            s += covered * 5000
            if covered == len(query_tokens):
                s += 12000

        downloads = int(model.get("downloads") or 0)
        likes = int(model.get("likes") or 0)
        s += min(downloads, 2_000_000) / 2000
        s += min(likes, 100_000) / 20
        return s

    return sorted(items, key=score, reverse=True)

def _hf_files(model):
    siblings = model.get("siblings") or []
    files = []
    allowed_ext = (
        ".safetensors",
        ".ckpt",
        ".pt",
        ".pth",
        ".bin",
        ".gguf",
        ".json",
        ".yaml",
        ".txt",
        ".model",
    )

    for s in siblings:
        name = s.get("rfilename") or s.get("filename") or ""
        if not name:
            continue

        # Put model-like files first, but still allow configs for full workflows.
        lower = name.lower()
        if lower.endswith(allowed_ext):
            files.append(name)

    def file_score(path):
        lower = path.lower()
        s = 0
        if lower.endswith(".safetensors"):
            s += 1000
        if lower.endswith(".gguf"):
            s += 900
        if lower.endswith(".ckpt"):
            s += 850
        if lower.endswith(".bin"):
            s += 600
        if "model" in lower or "diffusion" in lower or "transformer" in lower:
            s += 100
        return (-s, path)

    return sorted(files, key=file_score)

def _hf_repo_line(model, folder):
    repo_id = model.get("modelId") or model.get("id")
    label = _safe_label(repo_id.split("/")[-1] if repo_id else "Hugging Face repo")
    return f"repo | {folder} | {repo_id} | {label}"

def _hf_file_line(model, folder, file_path):
    repo_id = model.get("modelId") or model.get("id")
    label = _safe_label(f"{repo_id.split('/')[-1]} — {file_path.split('/')[-1]}")
    encoded_path = "/".join(urllib.parse.quote(p) for p in file_path.split("/"))
    return f"{folder} | https://huggingface.co/{repo_id}/resolve/main/{encoded_path} | {label}"

def _append_line_to_models(line):
    global MODELS

    line = line.strip()
    if not line:
        return

    current = MODELS.strip("\n")

    if line in current:
        return

    lines = current.splitlines()
    insert_at = 0

    for i, raw in enumerate(lines):
        stripped = raw.strip()
        if stripped and not stripped.startswith("#") and "|" in stripped:
            left, right = stripped.split("|", 1)
            if not right.strip():
                insert_at = i
                break
        insert_at = i + 1

    lines.insert(insert_at, line)
    MODELS = "\n".join(lines).strip("\n") + "\n"

# -------------------------
# UI
# -------------------------

source_tabs = widgets.ToggleButtons(
    options=[("CivitAI", "civitai"), ("Hugging Face", "hf")],
    value="civitai",
    description="Source:",
)

query_box = widgets.Text(
    value="",
    placeholder="Type model name OR paste model URL / repo ID",
    description="Search:",
    layout=widgets.Layout(width="75%")
)

search_button = widgets.Button(
    description="Search",
    button_style="primary",
)

folder_dropdown = widgets.Dropdown(
    options=[
        "checkpoints",
        "loras",
        "vae",
        "controlnet",
        "upscale_models",
        "diffusion_models",
        "text_encoders",
        "clip",
        "unet",
        "clip_vision",
        "embeddings",
        "diffusers",
    ],
    value="checkpoints",
    description="Folder:",
    layout=widgets.Layout(width="45%")
)

result_dropdown = widgets.Dropdown(
    options=[],
    description="Model:",
    layout=widgets.Layout(width="95%")
)

version_dropdown = widgets.Dropdown(
    options=[],
    description="Version:",
    layout=widgets.Layout(width="95%")
)

hf_download_mode = widgets.ToggleButtons(
    options=[("Full repo", "repo"), ("Single file", "file")],
    value="repo",
    description="HF mode:",
)

hf_file_dropdown = widgets.Dropdown(
    options=[],
    description="File:",
    layout=widgets.Layout(width="95%")
)

copy_box = widgets.Textarea(
    value="",
    description="Line:",
    layout=widgets.Layout(width="100%", height="70px"),
    disabled=False,
)

add_button = widgets.Button(
    description="Add selected model to MODELS",
    button_style="success",
)

show_models_button = widgets.Button(
    description="Show current MODELS",
    button_style="info",
)

status = widgets.Output()

STATE = {
    "civitai_results": [],
    "hf_results": [],
}

def _set_hf_controls_visible(is_hf):
    version_dropdown.layout.display = "none" if is_hf else None
    hf_download_mode.layout.display = None if is_hf else "none"
    hf_file_dropdown.layout.display = None if (is_hf and hf_download_mode.value == "file") else "none"

def _refresh_hf_files():
    if source_tabs.value != "hf" or result_dropdown.value is None or not STATE["hf_results"]:
        hf_file_dropdown.options = []
        return

    model = STATE["hf_results"][result_dropdown.value]
    files = _hf_files(model)

    if files:
        hf_file_dropdown.options = [(f, f) for f in files]
        hf_file_dropdown.value = files[0]
    else:
        hf_file_dropdown.options = []
        hf_file_dropdown.value = None

def _refresh_copy_line(*args):
    source = source_tabs.value
    folder = folder_dropdown.value

    if source == "civitai":
        if not result_dropdown.options or result_dropdown.value is None:
            copy_box.value = ""
            return
        model_index = result_dropdown.value
        model = STATE["civitai_results"][model_index]
        versions = _civitai_versions(model)
        if not versions or version_dropdown.value is None:
            copy_box.value = ""
            return
        version = versions[version_dropdown.value]
        copy_box.value = _civitai_copy_line(model, version, folder)

    else:
        if not result_dropdown.options or result_dropdown.value is None:
            copy_box.value = ""
            return
        model_index = result_dropdown.value
        model = STATE["hf_results"][model_index]

        if hf_download_mode.value == "repo":
            copy_box.value = _hf_repo_line(model, folder)
        else:
            file_path = hf_file_dropdown.value
            if not file_path:
                copy_box.value = ""
                return
            copy_box.value = _hf_file_line(model, folder, file_path)

def _on_source_change(change):
    with status:
        clear_output()
        print("Source changed. Type a search term and click Search.")

    result_dropdown.options = []
    version_dropdown.options = []
    hf_file_dropdown.options = []
    copy_box.value = ""

    is_hf = source_tabs.value == "hf"
    _set_hf_controls_visible(is_hf)

    if is_hf:
        folder_dropdown.value = "diffusers"
    else:
        folder_dropdown.value = "checkpoints"

def _on_search(_):
    query = query_box.value.strip()
    source = source_tabs.value

    with status:
        clear_output()
        if len(query) < 2:
            print("Type at least 2 characters, paste a CivitAI model URL/ID, or paste a Hugging Face repo URL/repo ID.")
            return
        print(f"Searching {'CivitAI' if source == 'civitai' else 'Hugging Face'} for: {query}")

    try:
        if source == "civitai":
            items = _civitai_search(query, limit=100)
            STATE["civitai_results"] = items

            if not items:
                with status:
                    clear_output()
                    print("No CivitAI results found.")
                    print("Tip: paste the CivitAI model page URL directly, like:")
                    print("https://civitai.com/models/376130/nova-anime-xl")
                return

            options = []
            for idx, model in enumerate(items):
                name = model.get("name") or "Unknown"
                model_id = model.get("id")
                mtype = model.get("type") or ""
                stats = _stats_text(model)
                options.append((f"{name} | {mtype} | model id {model_id} | {stats}", idx))

            result_dropdown.options = options
            result_dropdown.value = options[0][1]

            versions = _civitai_versions(items[0])
            _set_hf_controls_visible(False)
            if versions:
                version_dropdown.options = [(_version_label(items[0], v), i) for i, v in enumerate(versions)]
                version_dropdown.value = 0
            else:
                version_dropdown.options = []
                version_dropdown.value = None

            with status:
                clear_output()
                print(f"Found {len(items)} CivitAI results.")
                print("Exact name/slug matches are boosted to the top.")
                print("If the model still does not show, paste the CivitAI model page URL or model ID.")

        else:
            items = _hf_search(query, limit=100)
            STATE["hf_results"] = items

            if not items:
                with status:
                    clear_output()
                    print("No Hugging Face results found.")
                    print("Tip: paste the exact Hugging Face repo URL or repo ID, like:")
                    print("https://huggingface.co/Tongyi-MAI/Z-Image-Turbo")
                    print("Tongyi-MAI/Z-Image-Turbo")
                return

            options = []
            for idx, model in enumerate(items):
                repo_id = model.get("modelId") or model.get("id") or "Unknown"
                downloads = model.get("downloads") or 0
                likes = model.get("likes") or 0
                tags = model.get("tags") or []
                task = ""
                for tag in tags:
                    if isinstance(tag, str) and ("text-to-image" in tag or "image" in tag or "diffusers" in tag):
                        task = tag
                        break
                task_txt = f" | {task}" if task else ""
                options.append((f"{repo_id}{task_txt} | downloads {downloads:,} | likes {likes:,}", idx))

            result_dropdown.options = options
            result_dropdown.value = options[0][1]

            _set_hf_controls_visible(True)
            folder_dropdown.value = "diffusers"
            _refresh_hf_files()

            with status:
                clear_output()
                print(f"Found {len(items)} Hugging Face results.")
                print("Exact repo/name matches are boosted to the top.")
                print("Use Full repo for Diffusers-style repos.")
                print("Use Single file if you want one .safetensors/.gguf/etc file.")

        _refresh_copy_line()

    except Exception as e:
        with status:
            clear_output()
            print("Search failed:")
            print(e)

def _on_result_change(change):
    source = source_tabs.value

    if change["new"] is None:
        return

    if source == "civitai":
        model = STATE["civitai_results"][change["new"]]
        versions = _civitai_versions(model)
        if versions:
            version_dropdown.options = [(_version_label(model, v), i) for i, v in enumerate(versions)]
            version_dropdown.value = 0
        else:
            version_dropdown.options = []
            version_dropdown.value = None

    else:
        _refresh_hf_files()

    _refresh_copy_line()

def _on_hf_mode_change(change):
    _set_hf_controls_visible(source_tabs.value == "hf")
    _refresh_copy_line()

def _on_add(_):
    line = copy_box.value.strip()
    with status:
        clear_output()
        if not line:
            print("No line selected yet. Search and pick a model first.")
            return
        _append_line_to_models(line)
        print("✅ Added this line to MODELS:")
        print(line)
        print("\nNow run Step 2 — Check model links.")
        print("\nCurrent MODELS:")
        print(MODELS)

def _on_show_models(_):
    with status:
        clear_output()
        print("Current MODELS:")
        print(MODELS)

source_tabs.observe(_on_source_change, names="value")
search_button.on_click(_on_search)
result_dropdown.observe(_on_result_change, names="value")
version_dropdown.observe(lambda change: _refresh_copy_line(), names="value")
folder_dropdown.observe(lambda change: _refresh_copy_line(), names="value")
hf_download_mode.observe(_on_hf_mode_change, names="value")
hf_file_dropdown.observe(lambda change: _refresh_copy_line(), names="value")
add_button.on_click(_on_add)
show_models_button.on_click(_on_show_models)

_set_hf_controls_visible(False)

display(Markdown("""
### Model search

Search CivitAI or Hugging Face, choose the exact result, then click **Add selected model to MODELS**.

For CivitAI, you can also paste a model URL or model ID.

For Hugging Face, you can paste a repo URL or repo ID.

Examples:

`https://civitai.com/models/376130/nova-anime-xl`

`Tongyi-MAI/Z-Image-Turbo`
"""))

display(source_tabs)
display(widgets.HBox([query_box, search_button]))
display(result_dropdown)
display(version_dropdown)
display(hf_download_mode)
display(hf_file_dropdown)
display(folder_dropdown)
display(Markdown("### Copy/add line"))
display(copy_box)
display(widgets.HBox([add_button, show_models_button]))
display(status)

print("Type a model name, URL, repo ID, or model ID. Click Search. Choose the correct result. Then click Add.")

# 📝 Advanced optional — Manual model lines

You can skip this.

Use this only if you already know the exact links and want to type them manually.

Format examples:

```text
checkpoints | https://civitai.com/api/download/models/123456 | My Checkpoint
loras | https://civitai.com/models/123456/example-lora | My LoRA
vae | https://huggingface.co/user/model/resolve/main/vae.safetensors | My VAE
repo | diffusers | owner/repo | Full Hugging Face Repo
```

In [ ]:
#@title 📝 Optional — Add manual model lines { display-mode: "form" }

# Paste manual lines here only if you want.
# This will ADD to anything you selected in Step 1.
# Leave it empty if you used the search picker.

MANUAL_MODELS = """
"""

if "MODELS" not in globals():
    MODELS = ""

manual_lines = []

for raw in MANUAL_MODELS.splitlines():
    line = raw.strip()
    if not line or line.startswith("#"):
        continue
    manual_lines.append(line)

if manual_lines:
    existing = MODELS.strip()
    for line in manual_lines:
        if line not in existing:
            MODELS = (MODELS.strip() + "\n" + line).strip() + "\n"
    print("✅ Added manual lines to MODELS.")
else:
    print("No manual lines added. This is fine if you used Step 1 search.")

print("\nCurrent MODELS:")
print(MODELS if MODELS.strip() else "(empty)")

# ✅ Step 2 — Check model links

Run this after using Step 1.

This checks CivitAI links, Hugging Face file links, and Hugging Face repo lines before downloading.

In [ ]:
#@title ✅ Step 2 — Check model links { display-mode: "form" }

import re
import urllib.parse

COMFY_DIR = "/content/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"

KNOWN_FOLDERS = {
    "checkpoints", "loras", "vae", "controlnet", "upscale_models",
    "diffusion_models", "text_encoders", "clip", "unet", "clip_vision",
    "embeddings", "diffusers",
}

FOLDER_ALIASES = {
    "checkpoint": "checkpoints",
    "lora": "loras",
    "upscaler": "upscale_models",
    "upscale": "upscale_models",
    "text_encoder": "text_encoders",
    "textencoder": "text_encoders",
    "diffusion": "diffusion_models",
    "model": "checkpoints",
    "diffuser": "diffusers",
}

def normalize_folder_for_check(folder):
    folder = folder.strip()
    return FOLDER_ALIASES.get(folder, folder)

def clean_label(value):
    return str(value or "").replace("|", "-").strip()

def hf_repo_id_from_input(value):
    value = value.strip()

    if value.startswith("https://huggingface.co/"):
        parsed = urllib.parse.urlparse(value)
        parts = [p for p in parsed.path.split("/") if p]
        if len(parts) >= 2:
            return "/".join(parts[:2])
        return None

    if re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$", value):
        return value

    return None

def looks_like_hf_file_url(link):
    parsed = urllib.parse.urlparse(link)
    parts = [p for p in parsed.path.split("/") if p]

    if "huggingface.co" not in parsed.netloc:
        return False, "This is not a Hugging Face link."

    if "blob" not in parts and "resolve" not in parts:
        repo_id = hf_repo_id_from_input(link)
        suggestion = ""
        if repo_id:
            suggestion = f"\nUse this instead:\nrepo | diffusers | {repo_id} | {repo_id}"
        return False, (
            "This looks like a Hugging Face repo page, not a single file link. "
            "For full repos, use repo mode. For single files, open the file on "
            "Hugging Face and copy a URL containing /blob/ or /resolve/." + suggestion
        )

    marker = "blob" if "blob" in parts else "resolve"
    marker_index = parts.index(marker)

    if len(parts) < marker_index + 3:
        return False, "This Hugging Face link is missing the filename."

    filename = parts[-1]
    if "." not in filename:
        return False, (
            "This Hugging Face link does not look like it ends with a file name. "
            "A file link usually ends with .safetensors, .ckpt, .pt, .pth, .bin, or .gguf."
        )

    return True, ""

def parse_model_box(text):
    items = []
    errors = []
    warnings = []

    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        parts = [p.strip() for p in line.split("|")]

        # Full Hugging Face repo mode:
        # repo | folder | owner/repo | optional name
        if len(parts) >= 3 and parts[0].lower() in ["repo", "hf_repo", "huggingface_repo"]:
            _, folder, repo_input = parts[:3]
            display_name = clean_label(parts[3]) if len(parts) >= 4 else ""

            if not folder:
                errors.append(f"Line {line_number}: repo mode is missing folder. Use: repo | diffusers | owner/repo | name")
                continue

            if not repo_input:
                errors.append(f"Line {line_number}: repo mode is missing repo id. Use: repo | diffusers | owner/repo | name")
                continue

            folder = normalize_folder_for_check(folder)
            repo_id = hf_repo_id_from_input(repo_input)

            if not repo_id:
                errors.append(
                    f"Line {line_number}: Hugging Face repo problem. Use owner/repo format, like:\n"
                    "repo | diffusers | Tongyi-MAI/Z-Image-Turbo | Z-Image-Turbo"
                )
                continue

            if not display_name:
                display_name = repo_id

            if folder not in KNOWN_FOLDERS:
                warnings.append(
                    f"Line {line_number}: '{folder}' is not a common ComfyUI model folder. "
                    "It may still work if ComfyUI uses that folder."
                )

            items.append({
                "folder": folder,
                "repo_id": repo_id,
                "source": "hf_repo",
                "line": line_number,
                "display_name": display_name,
            })
            continue

        # Single file mode:
        # folder | link | optional name
        if "|" not in line:
            errors.append(f"Line {line_number}: missing | symbol. Use: folder | link | optional name")
            continue

        if len(parts) < 2:
            errors.append(f"Line {line_number}: use folder | link | optional name")
            continue

        folder, link = parts[0], parts[1]
        display_name = clean_label(parts[2]) if len(parts) >= 3 else ""

        if not link:
            continue

        if not folder:
            errors.append(f"Line {line_number}: folder is empty.")
            continue

        folder = normalize_folder_for_check(folder)

        if folder not in KNOWN_FOLDERS:
            warnings.append(
                f"Line {line_number}: '{folder}' is not a common ComfyUI model folder. "
                "It may still work if ComfyUI uses that folder."
            )

        link_lower = link.lower()

        if "civitai.com" in link_lower:
            source = "civitai"
        elif "huggingface.co" in link_lower:
            ok, message = looks_like_hf_file_url(link)
            if not ok:
                errors.append(f"Line {line_number}: Hugging Face link problem. {message}")
                continue
            source = "huggingface"
        else:
            errors.append(
                f"Line {line_number}: link is not CivitAI or Hugging Face. "
                "Use a civitai.com link, huggingface.co file link, or repo mode."
            )
            continue

        if not display_name:
            display_name = link.split("/")[-1] or link

        items.append({
            "folder": folder,
            "link": link,
            "source": source,
            "line": line_number,
            "display_name": display_name,
        })

    return items, errors, warnings

DOWNLOAD_ITEMS, PARSE_ERRORS, PARSE_WARNINGS = parse_model_box(MODELS)

print("✅ Link check finished")

if PARSE_WARNINGS:
    print("\n⚠️ Warnings:")
    for warning in PARSE_WARNINGS:
        print("-", warning)

if PARSE_ERRORS:
    print("\n❌ Fix these problems before continuing:")
    for err in PARSE_ERRORS:
        print("-", err)
    raise ValueError("Fix the link problems in Step 1, then run Step 2 again.")

print("\nItems found:")
if DOWNLOAD_ITEMS:
    for i, item in enumerate(DOWNLOAD_ITEMS, start=1):
        name = item.get("display_name", "")
        if item["source"] == "hf_repo":
            print(f"{i}. {name}  →  repo | {item['folder']} | {item['repo_id']}")
        else:
            print(f"{i}. {name}  →  {item['folder']} | {item['source']} | {item['link']}")
    print("\n✅ Good. Now run Step 3.")
else:
    print("No model links found yet.")
    print("\nExamples:")
    print("loras | https://civitai.com/models/123456/example-lora | Example LoRA")
    print("vae | https://huggingface.co/user/model/resolve/main/vae.safetensors | Example VAE")
    print("repo | diffusers | Tongyi-MAI/Z-Image-Turbo | Z-Image-Turbo")
    raise ValueError("No valid model links found.")

# 🔑 Step 3 — Tokens and settings

Run this after Step 2.

Tokens are hidden while typing.

Leave tokens empty if you do not need them.

In [ ]:
#@title 🔑 Step 3 — Tokens and settings { display-mode: "form" }

from getpass import getpass

#@markdown ## CivitAI token
#@markdown Turn this on if CivitAI downloads fail, if a model needs login, or if CivitAI returns access denied.
USE_CIVITAI_TOKEN = True #@param {type:"boolean"}

if USE_CIVITAI_TOKEN:
    CIVITAI_TOKEN = getpass("Paste your CivitAI API token. Leave empty if not needed: ").strip()
else:
    CIVITAI_TOKEN = ""

#@markdown ---
#@markdown ## Hugging Face token
#@markdown Turn this on only for private/gated Hugging Face files.
#@markdown Public Hugging Face files usually do not need this.
USE_HF_TOKEN = False #@param {type:"boolean"}

if USE_HF_TOKEN:
    HF_TOKEN = getpass("Paste your Hugging Face token. Leave empty if not needed: ").strip()
else:
    HF_TOKEN = ""

#@markdown ---
#@markdown ## VRAM mode
#@markdown `--lowvram` is recommended for free Colab T4.
#@markdown `--normalvram` is faster if you have enough VRAM.
#@markdown `--novram` is usually very slow; use only if lowvram fails.
VRAM_MODE = "--lowvram" #@param ["--lowvram", "--normalvram", "--novram"]

#@markdown ---
#@markdown ## Custom nodes
#@markdown ComfyUI Manager helps install/manage nodes inside ComfyUI.
#@markdown Impact Pack and Ultimate SD Upscale are useful for common workflows.
#@markdown GGUF node is useful for GGUF/quantized models.
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}
INSTALL_IMPACT_PACK = True #@param {type:"boolean"}
INSTALL_ULTIMATE_SD_UPSCALE = True #@param {type:"boolean"}
INSTALL_GGUF_NODE = True #@param {type:"boolean"}

print("✅ Settings saved")
print("CivitAI token provided:", bool(CIVITAI_TOKEN))
print("Hugging Face token provided:", bool(HF_TOKEN))
print("VRAM mode:", VRAM_MODE)
print("\nNext: run Step 4")

# ✅ Step 4 — Check GPU and storage

Run this after Step 3.

If this step says **No GPU found**, click:

**Runtime → Change runtime type → T4 GPU → Save**

In [ ]:
#@title ✅ Step 4 — Check GPU and storage { display-mode: "form" }

import subprocess, shutil

print("Checking GPU...")

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True
    )
    gpu_text = gpu.stdout.strip()
except Exception:
    gpu_text = ""

if not gpu_text:
    raise SystemExit(
        "❌ No GPU found.\n\n"
        "Fix:\n"
        "Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save\n"
        "Then run this step again."
    )

print("✅ GPU found:")
for line in gpu_text.splitlines():
    print("  " + line)

total, used, free = shutil.disk_usage("/content")

print("\nTemporary Colab storage:")
print(f"Total: {total / 1024**3:.1f} GB")
print(f"Free : {free / 1024**3:.1f} GB")

if free < 15 * 1024**3:
    print("\n⚠️ Warning: Less than 15 GB free. Big models may fail.")
else:
    print("\n✅ Storage looks okay.")

print("\nNext: run Step 5")

# 📦 Step 5 — Install ComfyUI

Run this step.

It installs ComfyUI and useful custom nodes.

In [ ]:
#@title 📦 Step 5 — Install ComfyUI { display-mode: "form" }

import os, subprocess

COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"

def run(cmd, cwd=None, quiet=False):
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        capture_output=quiet,
        text=True
    )

    if result.returncode != 0:
        if quiet:
            print(result.stdout[-1500:])
            print(result.stderr[-1500:])
        raise RuntimeError(f"Command failed: {cmd}")

print("Installing system tools...")
run("apt-get update -qq", quiet=True)
run("apt-get install -y -qq git wget aria2", quiet=True)

if not os.path.exists(COMFY_DIR):
    print("Installing ComfyUI...")
    run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI "{COMFY_DIR}"')
else:
    print("ComfyUI already exists. Updating...")
    run("git pull", cwd=COMFY_DIR, quiet=True)

print("Installing ComfyUI requirements...")
run(
    f'pip install -q -r "{COMFY_DIR}/requirements.txt" --extra-index-url https://download.pytorch.org/whl/cu121',
    quiet=True
)

print("Installing Hugging Face downloader...")
run("pip install -q huggingface_hub", quiet=True)

os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

nodes = []

if globals().get("INSTALL_COMFYUI_MANAGER", True):
    nodes.append(("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager"))

if globals().get("INSTALL_IMPACT_PACK", True):
    nodes.append(("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack"))

if globals().get("INSTALL_ULTIMATE_SD_UPSCALE", True):
    nodes.append(("ComfyUI_UltimateSDUpscale", "https://github.com/ssitu/ComfyUI_UltimateSDUpscale"))

if globals().get("INSTALL_GGUF_NODE", True):
    nodes.append(("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF"))

nodes.append(("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy"))

nodes.append(("Civicomfy", "https://github.com/MoonGoblinDev/Civicomfy"))

for name, repo in nodes:
    path = f"{CUSTOM_NODES_DIR}/{name}"

    if os.path.exists(path):
        print(f"✅ {name} already installed")
    else:
        print(f"Installing {name}...")
        run(f'git clone --depth 1 "{repo}" "{path}"', quiet=True)

for name, _ in nodes:
    req = f"{CUSTOM_NODES_DIR}/{name}/requirements.txt"

    if os.path.exists(req):
        print(f"Installing requirements for {name}...")
        run(f'pip install -q -r "{req}"', quiet=True)

print("\n✅ ComfyUI is installed")
print("Next: run Step 6")

# ⬇️ Step 6 — Download models

Run this step.

If any model fails, the notebook stops here so you do not accidentally launch ComfyUI with missing models.

In [ ]:
#@title ⬇️ Step 6 — Download models { display-mode: "form" }

import os, re, urllib.parse, requests
from tqdm.auto import tqdm

COMFY_DIR = "/content/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"

CIVITAI_TOKEN = globals().get("CIVITAI_TOKEN", "")
HF_TOKEN = globals().get("HF_TOKEN", "")

def item_display_name(item):
    return item.get("display_name") or item.get("repo_id") or item.get("link") or "Unnamed item"


ALLOWED_FOLDERS = {
    "checkpoints", "loras", "vae", "controlnet", "upscale_models",
    "diffusion_models", "text_encoders", "clip", "unet", "clip_vision",
    "embeddings", "diffusers",
}

def safe_name(name):
    name = str(name or "").strip().replace("\\", "_").replace("/", "_")
    name = re.sub(r"[^a-zA-Z0-9._() \-+\[\]]+", "_", name)
    return name[:180] or "downloaded_model.safetensors"

def safe_repo_dir(repo_id):
    return repo_id.replace("/", "--").replace("\\", "--")

def filename_from_header(header):
    if not header:
        return None
    match = re.search(r"filename\*=UTF-8''([^;]+)", header, flags=re.I)
    if match:
        return safe_name(urllib.parse.unquote(match.group(1)))
    match = re.search(r'filename="?([^";]+)"?', header, flags=re.I)
    if match:
        return safe_name(match.group(1))
    return None

def normalize_folder(folder):
    folder = folder.strip()
    aliases = {
        "checkpoint": "checkpoints", "lora": "loras",
        "upscaler": "upscale_models", "upscale": "upscale_models",
        "text_encoder": "text_encoders", "textencoder": "text_encoders",
        "diffusion": "diffusion_models", "model": "checkpoints",
        "diffuser": "diffusers",
    }
    return aliases.get(folder, folder)

def stream_download(url, target_dir, filename, headers=None, params=None):
    headers = headers or {}
    params = params or {}
    os.makedirs(target_dir, exist_ok=True)
    filename = safe_name(filename)
    final_path = f"{target_dir}/{filename}"
    part_path = final_path + ".part"

    if os.path.exists(final_path) and os.path.getsize(final_path) > 1_000_000:
        print("✅ Already downloaded, skipping:")
        print(final_path)
        return final_path

    existing = os.path.getsize(part_path) if os.path.exists(part_path) else 0
    request_headers = dict(headers)
    if existing > 0:
        request_headers["Range"] = f"bytes={existing}-"
        print(f"Resuming from {existing / 1024**2:.1f} MB")

    with requests.get(url, params=params, headers=request_headers, stream=True, allow_redirects=True, timeout=60) as response:
        print(f"HTTP status: {response.status_code}")
        print(f"Content type: {response.headers.get('content-type', '')}")
        if response.status_code in [401, 403]:
            raise RuntimeError("Download rejected. Token may be missing/invalid, or access is not granted.")
        if response.status_code == 404:
            raise RuntimeError("Download returned 404. Link may be wrong or unavailable.")
        if response.status_code not in [200, 206]:
            raise RuntimeError(f"Download failed: HTTP {response.status_code}\n{response.text[:700]}")

        content_type = response.headers.get("content-type", "").lower()
        if "text/html" in content_type:
            raise RuntimeError("Server returned HTML instead of a model file. This usually means login/token/link issue.")
        if "application/json" in content_type:
            try:
                data = response.json()
            except Exception:
                data = response.text[:700]
            raise RuntimeError(f"Server returned JSON instead of a model file:\n{data}")

        header_filename = filename_from_header(response.headers.get("content-disposition", ""))
        if header_filename and existing == 0:
            filename = header_filename
            final_path = f"{target_dir}/{filename}"
            part_path = final_path + ".part"
            print(f"Filename detected: {filename}")

        if existing > 0 and response.status_code == 200:
            existing = 0
            mode = "wb"
        else:
            mode = "ab" if existing > 0 else "wb"

        total = response.headers.get("content-length")
        total = int(total) if total and total.isdigit() else None
        progress_total = total + existing if total else None
        with open(part_path, mode) as file, tqdm(total=progress_total, initial=existing, unit="B", unit_scale=True, unit_divisor=1024) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    bar.update(len(chunk))

    if not os.path.exists(part_path):
        raise RuntimeError("Download ended, but file was not created.")
    size = os.path.getsize(part_path)
    if size < 1_000_000:
        raise RuntimeError(f"Downloaded file is too small: {size} bytes. Probably an error page.")
    os.replace(part_path, final_path)
    print("✅ Done")
    print(final_path)
    print(f"Size: {os.path.getsize(final_path) / 1024**3:.2f} GB")
    return final_path

def extract_civitai_kind_and_id(value):
    value = value.strip()
    match = re.search(r"/api/download/models/(\d+)", value)
    if match:
        return "version", match.group(1)
    match = re.search(r"/models/(\d+)", value)
    if match:
        return "model", match.group(1)
    if value.isdigit():
        return "unknown_number", value
    raise ValueError("Paste a CivitAI model page URL, download-button URL, or numeric ID.")

def civitai_get_json(url, token=""):
    params = {"token": token} if token else {}
    headers = {"User-Agent": "Mozilla/5.0 Colab ComfyUI Public Notebook", "Accept": "application/json"}
    response = requests.get(url, params=params, headers=headers, timeout=30)
    if response.status_code != 200:
        print(f"⚠️ Metadata failed: HTTP {response.status_code}")
        print(response.text[:700])
        return None
    try:
        return response.json()
    except Exception:
        print("⚠️ Metadata response was not JSON")
        print(response.text[:700])
        return None

def choose_civitai_version_from_model(model_data):
    versions = model_data.get("modelVersions") or []
    if not versions:
        raise RuntimeError("No model versions found on this CivitAI page.")
    print("\nAvailable versions:")
    for i, v in enumerate(versions):
        print(f"  {i}: {v.get('name')} | version id: {v.get('id')}")
    chosen = versions[0]
    print(f"\n✅ Auto-selected latest/first version: {chosen.get('name')}")
    print("Tip: To force another version, paste that version's Download button link.")
    return chosen

def resolve_civitai_version(civitai_input, token):
    kind, number = extract_civitai_kind_and_id(civitai_input)
    if kind == "version":
        version_data = civitai_get_json(f"https://civitai.com/api/v1/model-versions/{number}", token)
        return number, version_data
    if kind == "model":
        print(f"Reading CivitAI page: model id {number}")
        model_data = civitai_get_json(f"https://civitai.com/api/v1/models/{number}", token)
        if not model_data:
            raise RuntimeError("Could not read CivitAI model metadata.")
        chosen = choose_civitai_version_from_model(model_data)
        return str(chosen["id"]), chosen
    if kind == "unknown_number":
        print(f"Trying number as version ID: {number}")
        version_data = civitai_get_json(f"https://civitai.com/api/v1/model-versions/{number}", token)
        if version_data and version_data.get("id"):
            return number, version_data
        print("Trying number as model ID...")
        model_data = civitai_get_json(f"https://civitai.com/api/v1/models/{number}", token)
        if not model_data:
            raise RuntimeError("Number did not work as version ID or model ID.")
        chosen = choose_civitai_version_from_model(model_data)
        return str(chosen["id"]), chosen

def choose_civitai_file(version_data):
    if not version_data:
        return None
    files = version_data.get("files") or []
    if not files:
        return None
    primary = [f for f in files if f.get("primary")]
    candidates = primary or files
    candidates = sorted(candidates, key=lambda f: f.get("sizeKB") or 0, reverse=True)
    chosen = candidates[0]
    if chosen.get("name"):
        print(f"File: {chosen.get('name')}")
    if chosen.get("sizeKB"):
        print(f"Expected size: {chosen.get('sizeKB') / 1024 / 1024:.2f} GB")
    return chosen

def download_civitai(item):
    folder = normalize_folder(item["folder"])
    link = item["link"]
    target_dir = f"{MODELS_DIR}/{folder}"
    print("\n" + "=" * 80)
    print(f"CivitAI download → models/{folder}")
    print(link)
    print("=" * 80)
    version_id, version_data = resolve_civitai_version(link, CIVITAI_TOKEN)
    if version_data is None:
        version_data = civitai_get_json(f"https://civitai.com/api/v1/model-versions/{version_id}", CIVITAI_TOKEN)
    chosen_file = choose_civitai_file(version_data)
    filename = safe_name((chosen_file or {}).get("name") or f"civitai_{version_id}.safetensors")
    url = f"https://civitai.com/api/download/models/{version_id}"
    params = {"token": CIVITAI_TOKEN} if CIVITAI_TOKEN else {}
    headers = {"User-Agent": "Mozilla/5.0 Colab ComfyUI Public Notebook", "Accept": "application/octet-stream, application/json, text/plain, */*"}
    return stream_download(url, target_dir, filename, headers=headers, params=params)

def parse_hf_url(url):
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    if "huggingface.co" not in parsed.netloc:
        raise ValueError("This does not look like a Hugging Face URL.")
    parts = [p for p in parsed.path.split("/") if p]
    marker_index = None
    for possible in ["blob", "resolve"]:
        if possible in parts:
            marker_index = parts.index(possible)
            break
    if marker_index is None:
        raise ValueError("Hugging Face link must point directly to a file and include /blob/ or /resolve/. For full repos, use: repo | diffusers | owner/repo")
    if len(parts) < marker_index + 3:
        raise ValueError("Hugging Face file link is incomplete.")
    repo_id = "/".join(parts[:marker_index])
    revision = parts[marker_index + 1]
    file_parts = parts[marker_index + 2:]
    file_path = "/".join(file_parts)
    filename = safe_name(file_parts[-1])
    resolve_url = f"https://huggingface.co/{repo_id}/resolve/{revision}/{file_path}"
    return resolve_url, filename, repo_id, file_path

def download_huggingface_file(item):
    folder = normalize_folder(item["folder"])
    link = item["link"]
    target_dir = f"{MODELS_DIR}/{folder}"
    print("\n" + "=" * 80)
    print(f"Hugging Face file download → models/{folder}")
    print(link)
    print("=" * 80)
    url, filename, repo_id, file_path = parse_hf_url(link)
    print(f"Repo: {repo_id}")
    print(f"File: {file_path}")
    headers = {"User-Agent": "Mozilla/5.0 Colab ComfyUI Public Notebook", "Accept": "application/octet-stream, application/json, text/plain, */*"}
    if HF_TOKEN:
        headers["Authorization"] = f"Bearer {HF_TOKEN}"
    return stream_download(url, target_dir, filename, headers=headers, params={})

def download_huggingface_repo(item):
    from huggingface_hub import snapshot_download
    folder = normalize_folder(item["folder"])
    repo_id = item["repo_id"]
    repo_dir_name = safe_repo_dir(repo_id)
    target_dir = f"{MODELS_DIR}/{folder}/{repo_dir_name}"
    print("\n" + "=" * 80)
    print(f"Hugging Face repo download → models/{folder}/{repo_dir_name}")
    print(f"Repo: {repo_id}")
    print("=" * 80)
    if os.path.exists(target_dir) and any(os.scandir(target_dir)):
        print("✅ Repo folder already exists, skipping:")
        print(target_dir)
        return target_dir
    token = HF_TOKEN if HF_TOKEN else None
    snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=target_dir,
        token=token,
        resume_download=True,
    )
    print("✅ Repo download complete")
    print(target_dir)
    return target_dir

if "DOWNLOAD_ITEMS" not in globals() or not DOWNLOAD_ITEMS:
    raise ValueError("No valid models found. Go back to Step 1 and Step 2 first.")

print(f"Found {len(DOWNLOAD_ITEMS)} item(s).")

downloaded = []
failed = []
for i, item in enumerate(DOWNLOAD_ITEMS, start=1):
    print(f"\nStarting item {i}/{len(DOWNLOAD_ITEMS)}: {item_display_name(item)}")
    try:
        if item["source"] == "civitai":
            downloaded.append(download_civitai(item))
        elif item["source"] == "huggingface":
            downloaded.append(download_huggingface_file(item))
        elif item["source"] == "hf_repo":
            downloaded.append(download_huggingface_repo(item))
        else:
            raise ValueError(f"Unknown source: {item['source']}")
    except Exception as e:
        print("\n❌ Failed:")
        print("Name:", item_display_name(item))
        if item["source"] == "hf_repo":
            print("repo |", item["folder"], "|", item["repo_id"])
        else:
            print(item["folder"], "|", item["link"])
        print("\nError:")
        print(e)
        failed.append((item, str(e)))

print("\n" + "=" * 80)
print("DOWNLOAD SUMMARY")
print("=" * 80)
if downloaded:
    print("\n✅ Downloaded:")
    for path in downloaded:
        print(" -", path)
if failed:
    print("\n❌ Failed:")
    for item, error in failed:
        print(" -", item_display_name(item))
        if item["source"] == "hf_repo":
            print("   repo |", item["folder"], "|", item["repo_id"])
        else:
            print("  ", item["folder"], "|", item["link"])
        print("   ", error)
if failed:
    raise RuntimeError("One or more downloads failed. Fix the failed links/tokens above, then run Step 6 again. ComfyUI was not started so you do not end up with missing models.")
if not downloaded:
    raise RuntimeError("No files were downloaded. Go back to Step 1, paste at least one valid model/file link, then rerun Step 2 and Step 6.")
print("\n🎉 All downloads finished successfully.")
print("Next: run Step 7")

# ▶️ Step 7 — Start ComfyUI

Run this after downloads finish.

In [ ]:
#@title ▶️ Step 7 — Start ComfyUI { display-mode: "form" }

import subprocess, requests, time

COMFY_DIR = "/content/ComfyUI"
VRAM_MODE = globals().get("VRAM_MODE", "--lowvram")

def comfy_is_alive():
    try:
        requests.get("http://127.0.0.1:8188", timeout=2)
        return True
    except Exception:
        return False

if comfy_is_alive():
    print("✅ ComfyUI is already running")
else:
    print("Starting ComfyUI...")

    subprocess.run(["pkill", "-f", "python main.py"], capture_output=True)
    time.sleep(2)

    cmd = [
        "python",
        "main.py",
        "--listen",
        "0.0.0.0",
        "--port",
        "8188",
    ]

    if VRAM_MODE and VRAM_MODE != "--normalvram":
        cmd.append(VRAM_MODE)

    print("Mode:", VRAM_MODE)

    log = open("/tmp/comfyui.log", "w")

    subprocess.Popen(
        cmd,
        cwd=COMFY_DIR,
        stdout=log,
        stderr=subprocess.STDOUT
    )

    print("Waiting for ComfyUI to start...")

    for i in range(120):
        if comfy_is_alive():
            print(f"\n✅ ComfyUI is ready after {i * 2} seconds")
            break

        print(".", end="", flush=True)
        time.sleep(2)
    else:
        print("\n❌ ComfyUI did not start.")
        print("Run the optional logs cell at the bottom.")

print("\nNext: run Step 8")

# 🌐 Step 8 — Get your ComfyUI link

Run this step.

Open the printed link in **Chrome** if possible.

If Firefox opens the page but ComfyUI does not load correctly, try Chrome or another Chromium-based browser.

In [ ]:
#@title 🌐 Step 8 — Get public link { display-mode: "form" }

import subprocess, re, time, requests

def comfy_is_alive():
    try:
        requests.get("http://127.0.0.1:8188", timeout=2)
        return True
    except Exception:
        return False

if not comfy_is_alive():
    raise RuntimeError("ComfyUI is not running yet. Run Step 7 first.")

print("Installing tunnel tool...")
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared",
    shell=True
)
subprocess.run("chmod +x /usr/local/bin/cloudflared", shell=True)

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

print("Creating public link...")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None

for _ in range(120):
    line = tunnel.stdout.readline()

    if line:
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

    time.sleep(0.5)

if not public_url:
    raise RuntimeError("Could not create Cloudflare link. Try running this step again.")

print("\n" + "=" * 90)
print("✅ COMFYUI IS READY")
print("=" * 90)
print("\nOpen this link:")
print(public_url)
print("\nKeep this Colab tab open while using ComfyUI.")
print("Browser tip: if Firefox does not load the ComfyUI page correctly, try Chrome/Chromium.")
print("=" * 90)

# 🧾 Optional — If something breaks

Use these only when needed.

In [ ]:
#@title 🧾 Optional — Show ComfyUI logs { display-mode: "form" }

print("Last 120 lines of ComfyUI log:\n")
!tail -120 /tmp/comfyui.log

In [ ]:
#@title 📁 Optional — Show downloaded files { display-mode: "form" }

import os

base = "/content/ComfyUI/models"

if not os.path.exists(base):
    print("No model folder found yet.")
else:
    found = False

    for root, dirs, files in os.walk(base):
        files = [
            f for f in files
            if not f.endswith(".part") and not f.startswith(".")
        ]

        if files:
            found = True
            print("\n📁", root)

            for f in files:
                path = os.path.join(root, f)
                size = os.path.getsize(path) / 1024**3
                print(f"  - {f} ({size:.2f} GB)")

    if not found:
        print("No downloaded model files found yet.")